# 📊 Exploratory Data Analysis
## Multilingual Health Question Answering in Low-Resource African Languages
**MLT1 Final Project | African Leadership University | June 2026**

This notebook performs a comprehensive EDA on the competition dataset to understand:
- Language and subset distribution
- Input/output text length characteristics
- Data quality (duplicates, missing values)
- Sample Q&A pairs per language
- Key preprocessing insights


In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 12,
})

SEED = 42
np.random.seed(SEED)
print("✅ Libraries loaded successfully")


In [ ]:
# ── Load Data ────────────────────────────────────────────────────
train = pd.read_csv('data/raw/Train.csv')
test  = pd.read_csv('data/raw/Test.csv')
sample_sub = pd.read_csv('data/raw/SampleSubmission.csv')

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Sample sub  : {sample_sub.shape}")
print()
print("Train columns:", train.columns.tolist())
print("Test columns :", test.columns.tolist())
print()
train.head(3)


In [ ]:
# ── Data Quality Check ───────────────────────────────────────────
print("=== Missing Values (Train) ===")
print(train.isnull().sum())
print()
print("=== Missing Values (Test) ===")
print(test.isnull().sum())
print()
print(f"Duplicate input questions  : {train['input'].duplicated().sum()}")
print(f"Duplicate output answers   : {train['output'].duplicated().sum()}")
print()
print("✅ No missing values — clean dataset!")


In [ ]:
# ── Language (Subset) Distribution ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Language labels mapping
lang_labels = {
    'Aka_Gha': 'Akan\n(Ghana)',
    'Amh_Eth': 'Amharic\n(Ethiopia)',
    'Eng_Eth': 'English\n(Ethiopia)',
    'Eng_Gha': 'English\n(Ghana)',
    'Eng_Ken': 'English\n(Kenya)',
    'Eng_Uga': 'English\n(Uganda)',
    'Lug_Uga': 'Luganda\n(Uganda)',
    'Swa_Ken': 'Kiswahili\n(Kenya)',
}

colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4','#FF5722','#8BC34A']

# Train distribution
train_counts = train['subset'].value_counts()
axes[0].bar([lang_labels[k] for k in train_counts.index], train_counts.values, color=colors)
axes[0].set_title('Training Set — Language Distribution', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', labelsize=9)
for i, v in enumerate(train_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=9, fontweight='bold')

# Test distribution
test_counts = test['subset'].value_counts()
axes[1].bar([lang_labels[k] for k in test_counts.index], test_counts.values, color=colors)
axes[1].set_title('Test Set — Language Distribution', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Number of Samples')
axes[1].tick_params(axis='x', labelsize=9)
for i, v in enumerate(test_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('results/eda_language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⚠️  Key insight: Dataset is imbalanced — Eng_Uga dominates (7,624 samples)")
print("⚠️  Amh_Eth has fewest samples (1,845 train / 61 test) — risk of lower performance")


In [ ]:
# ── Text Length Analysis ─────────────────────────────────────────
train['input_word_len']  = train['input'].str.split().str.len()
train['output_word_len'] = train['output'].str.split().str.len()
train['input_char_len']  = train['input'].str.len()
train['output_char_len'] = train['output'].str.len()

print("=== Input (Question) Word Count Stats ===")
print(train['input_word_len'].describe().round(1))
print()
print("=== Output (Answer) Word Count Stats ===")
print(train['output_word_len'].describe().round(1))


In [ ]:
# ── Length Distribution Plots ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Input word length
axes[0,0].hist(train['input_word_len'], bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0,0].axvline(train['input_word_len'].mean(), color='red', linestyle='--', label=f"Mean: {train['input_word_len'].mean():.1f}")
axes[0,0].set_title('Question Word Count Distribution', fontweight='bold')
axes[0,0].set_xlabel('Word Count'); axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()

# Output word length
axes[0,1].hist(train['output_word_len'], bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[0,1].axvline(train['output_word_len'].mean(), color='red', linestyle='--', label=f"Mean: {train['output_word_len'].mean():.1f}")
axes[0,1].set_title('Answer Word Count Distribution', fontweight='bold')
axes[0,1].set_xlabel('Word Count'); axes[0,1].set_ylabel('Frequency')
axes[0,1].legend()

# Per-language avg input length
lang_stats = train.groupby('subset')[['input_word_len','output_word_len']].mean().round(1)
lang_stats.index = [lang_labels[i] for i in lang_stats.index]

axes[1,0].barh(lang_stats.index, lang_stats['input_word_len'], color='#2196F3', alpha=0.85)
axes[1,0].set_title('Avg Question Length per Language', fontweight='bold')
axes[1,0].set_xlabel('Avg Word Count')
for i, v in enumerate(lang_stats['input_word_len']):
    axes[1,0].text(v + 0.3, i, str(v), va='center', fontsize=9)

# Per-language avg output length
axes[1,1].barh(lang_stats.index, lang_stats['output_word_len'], color='#4CAF50', alpha=0.85)
axes[1,1].set_title('Avg Answer Length per Language', fontweight='bold')
axes[1,1].set_xlabel('Avg Word Count')
for i, v in enumerate(lang_stats['output_word_len']):
    axes[1,1].text(v + 0.3, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/eda_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⚠️  Key insight: Akan (Aka_Gha) has the longest answers (~106 words) — hardest generation task")
print("⚠️  Amharic answers are shortest (~20 words) — but model may still struggle due to script complexity")


In [ ]:
# ── Per-Language Summary Table ───────────────────────────────────
summary = train.groupby('subset').agg(
    Train_Samples=('ID', 'count'),
    Avg_Q_Words=('input_word_len', 'mean'),
    Avg_A_Words=('output_word_len', 'mean'),
    Max_Q_Words=('input_word_len', 'max'),
    Max_A_Words=('output_word_len', 'max'),
).round(1)

test_counts_df = test['subset'].value_counts().rename('Test_Samples')
summary = summary.join(test_counts_df)
summary = summary[['Train_Samples','Test_Samples','Avg_Q_Words','Avg_A_Words','Max_Q_Words','Max_A_Words']]
summary.index = [lang_labels[i].replace('\n',' ') for i in summary.index]
print(summary.to_string())


In [ ]:
# ── Sample Q&A Pairs per Language ────────────────────────────────
print("=" * 70)
for lang in sorted(train['subset'].unique()):
    label = lang_labels[lang].replace('\n', ' ')
    row = train[train['subset'] == lang].iloc[0]
    print(f"\n🌍 [{label}] — subset: {lang}")
    print(f"❓ Q: {row['input'][:200]}")
    print(f"💬 A: {row['output'][:200]}")
    print("-" * 70)


In [ ]:
# ── Duplicate Analysis ───────────────────────────────────────────
dup_inputs  = train[train['input'].duplicated(keep=False)]
dup_outputs = train[train['output'].duplicated(keep=False)]

print(f"Duplicate input questions : {train['input'].duplicated().sum()} ({train['input'].duplicated().mean()*100:.1f}%)")
print(f"Duplicate output answers  : {train['output'].duplicated().sum()} ({train['output'].duplicated().mean()*100:.1f}%)")
print()
print("Duplicates per language (input):")
print(dup_inputs['subset'].value_counts())
print()
print("⚠️  High answer duplication (~39%) suggests some health topics have standardised responses")
print("✅  This means the model needs to learn fluent reuse, not always novel generation")


In [ ]:
# ── Input/Output Length Ratio ────────────────────────────────────
train['io_ratio'] = train['output_word_len'] / train['input_word_len'].replace(0, np.nan)

print("Answer-to-Question length ratio (words):")
print(train['io_ratio'].describe().round(2))
print()

fig, ax = plt.subplots(figsize=(10, 4))
train.groupby('subset')['io_ratio'].mean().sort_values().plot(
    kind='barh', ax=ax, color='#9C27B0', alpha=0.85
)
ax.set_title('Avg Answer/Question Word Ratio per Language', fontweight='bold')
ax.set_xlabel('Ratio (Answer words / Question words)')
ax.set_yticklabels([lang_labels[i.get_text()].replace('\n',' ') for i in ax.get_yticklabels()])
plt.tight_layout()
plt.savefig('results/eda_io_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⚠️  Models need to learn to expand short questions into long detailed answers")


## 🔑 Key EDA Insights & Preprocessing Decisions

| Insight | Impact on Modeling |
|---|---|
| 8 language subsets (4 African, 4 English variants) | Need multilingual model (mT5/AfroLM) |
| Dataset is **imbalanced** — Eng_Uga dominates | Monitor per-language ROUGE scores separately |
| Amharic has fewest samples (1,845) | May underperform — consider data augmentation |
| Questions: avg ~15 words, Answers: avg ~76 words | Set `max_input_length=128`, `max_target_length=256` |
| Akan answers longest (~106 words) | May need `max_target_length=512` for Akan |
| ~39% duplicate answers | Standardised health responses — model can leverage this |
| No missing values | No imputation needed |
| Scripts: Latin (English, Akan), Ethiopic (Amharic), Latin (Luganda, Kiswahili) | Tokenizer must handle multiple scripts |

### Preprocessing Plan
1. Strip leading/trailing whitespace
2. Normalize Unicode characters (especially Akan and Amharic scripts)
3. Add language-aware prompt prefix: `"answer health question in {language}: {question}"`
4. Tokenize with mT5 tokenizer (supports all languages above)
5. Set `max_input_length=128`, `max_target_length=256` (tune per experiment)
6. Train/validation split: 90/10 stratified by `subset`
